# Exercise 7.3. Evaluation and plotting

The aim of this notebook is to:
* evaluate the model's accuracy
* plot results as map

The task is land-use classification, based on Sentinel-2 11-days mosaic.

**This notebook should be run twice (with own CNN model and foundation-model based CNN model)**

## Results
For each model (2 models)
* Model accuracy estimation
* Class confusion matrix
* Predicted classification as GeoTIFFs

Models:
* Sentinel-2 + own CNN-model
* Sentinel-2 + foundation-model based CNN-model 

In [ ]:
import os, time
from joblib import dump, load
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np
import pandas as pd
import rasterio
from rasterio.plot import show
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, jaccard_score
from sklearn.model_selection import GridSearchCV
%matplotlib inline

In [ ]:
# Set folders and file paths
user = os.environ.get('USER')
base_folder = os.path.join('/scratch/project_2019932/students', user, 'GeoML')
dataFolder = os.path.join(base_folder,'data', 'raster', 'pixel-wise')
results_folder = os.path.join(base_folder, "classification_results")
classification_report_csv = os.path.join(results_folder, "classification_report_results.csv")
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

dataset = "sentinel2"
#TODO, change here the model
model = "cnn_own"
#model = "cnn_fm" 

# Saved test files from exercise 2.
data_file = os.path.join(dataFolder, 'data_sentinel2.tif')
labels_file = os.path.join(dataFolder, 'labels.tif')

# Paths for GeoTIFFs created with the prediction Python code:
# Most likely class
predicted_classification = os.path.join(results_folder, ('classification_' + dataset + '_' + model +  '.tif'))

# Probabilities for all classes
probabilities_all_classes = os.path.join(results_folder, ('class_probabilities_' + dataset + '_' + model +  '_all_classes.tif'))

In [ ]:
#Give classes and bands names for plotting
class_names = ["other", "forest", "fields", "water"]
number_of_classes = len(class_names)

band_names = ["b02", "b03", "b04", "b05", "b06", "b07", "b08", "b8a", "b11", "b12"]

### Load the test data from the saved files.

In [ ]:
with rasterio.open(labels_file) as src:
    ground_truth = src.read().reshape(-1)

with rasterio.open(predicted_classification) as src:
    predictions = src.read().reshape(-1)    

### Classification reports

In [ ]:
print('Classification report: \n', classification_report(ground_truth, predictions, target_names=class_names))

Save the results for later comparison.

In [ ]:
classification_report_results = classification_report(ground_truth, predictions, output_dict=True)

# Convert to Pandas dataframe
df = pd.json_normalize(classification_report_results)
df.insert(0, "dataset", dataset)
df.insert(1, "model", model)

### Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    ground_truth,
    predictions,
    normalize='true',
    cmap=plt.cm.Blues,
    display_labels=class_names,
    colorbar=False
)

### Jaccard score

In [ ]:
jaccard_score_value = round(jaccard_score(ground_truth, predictions, average='macro'), 3)
print(jaccard_score_value, model, dataset)
# Save the values to the dataframe
mask = (df['dataset'] == dataset) & (df['model'] == model)
df.loc[mask, 'jaccard'] = jaccard_score_value

### Save model metrics

In [ ]:
# Add this model to the metircs table
if os.path.exists(classification_report_csv):
    df_old =  pd.read_csv(classification_report_csv)
    df = pd.concat([df_old, df], ignore_index=True)

In [ ]:
# Have a look on the dataframe
df[["dataset","model","accuracy", "jaccard","macro avg.precision", "macro avg.recall", "macro avg.f1-score", "0.f1-score", "1.f1-score","2.f1-score", "3.f1-score"]]

In [ ]:
# Save the dataframe
df.to_csv(classification_report_csv, index=False)

## Plot result maps

#### Class-wise probabilites as map

Plot the probabilieis for all classes, most likely class and ground truth.

Compare the probabilities for different classes.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 30), constrained_layout=True)
cmap = ListedColormap(["black", "forestgreen", "lightyellow", "lightblue"])

with rasterio.open(probabilities_all_classes) as src:
    data = src.read()
    transform = src.transform  # reuse this so top panels match bottom panels' coordinate space

    for i, ax in enumerate(axes.flat[:4]):
        im = show(data[i], ax=ax, cmap="gray", title=class_names[i], transform=transform)
        mappable = ax.images[0]  # grab the actual image artist for the colorbar
        ax.set_aspect("equal")
        fig.colorbar(mappable, ax=ax, fraction=0.046, pad=0.04)

with rasterio.open(predicted_classification) as src:
    show(src, ax=axes[2, 0], cmap=cmap, title='Predicted classes')
    axes[2, 0].set_aspect("equal")

with rasterio.open(labels_file) as labels:
    show(labels, ax=axes[2, 1], cmap=cmap, title='Labels')
    axes[2, 1].set_aspect("equal")

plt.show()

### Predicted classification

In [ ]:
# For better plotting of Sentinel image, normalize the values
# Help function to normalize band values and enhance contrast. Just like what QGIS does automatically
def normalize(rgb):
    min_percent = 2   # Low percentile
    max_percent = 98  # High percentile
    lo, hi = np.percentile(rgb, (min_percent, max_percent), axis=(0,1), keepdims=True)
    new_min, new_max = 1, 255
    rgb_norm = (rgb - lo) / (hi - lo) * (new_max - new_min) + new_min
    rgb_norm = rgb_norm.astype(np.uint8)    
    return rgb_norm.astype(np.uint8)

In [ ]:

datasets = ["sentinel2", "alphaearth"]
models = ["random_forest", "hist_gradient_boosting", "mlp", "cnn_own", "cnn_fm"]

# Paths to all classification results we have got during the course
predicted_classification_files = {
    (datasets[0],models[0]): os.path.join(results_folder, ('classification_' + datasets[0] + '_' + models[0] +  '.tif')),
    (datasets[0],models[1]): os.path.join(results_folder, ('classification_' + datasets[0] + '_' + models[1] +  '.tif')),
    (datasets[1],models[0]): os.path.join(results_folder, ('classification_' + datasets[1] + '_' + models[0] +  '.tif')),    
    (datasets[1],models[1]): os.path.join(results_folder, ('classification_' + datasets[1] + '_' + models[1] +  '.tif')),
    (datasets[0],models[2]): os.path.join(results_folder, ('classification_' + datasets[0] + '_' + models[2] +  '.tif')),
    (datasets[1],models[2]): os.path.join(results_folder, ('classification_' + datasets[1] + '_' + models[2] +  '.tif')),    
    (datasets[0],models[3]): os.path.join(results_folder, ('classification_' + datasets[0] + '_' + models[3] +  '.tif')),    
    (datasets[0],models[4]): os.path.join(results_folder, ('classification_' + datasets[0] + '_' + models[4] +  '.tif')),
}

In [ ]:
predicted_classification_files

In [ ]:
# Create a subplot for 6 images: 4 classification, 1 data image and 1 training labels. 
fig, axes = plt.subplots(ncols=2, nrows=5, figsize=(20,40))
for ax, (key, path) in zip(axes.flat, predicted_classification_files.items()):
    dataset, model = key
    title = f"{dataset} {model}"
    if os.path.exists(predicted_classification_files[(dataset, model)]):
        with rasterio.open(predicted_classification_files[(dataset, model)]) as src:
            show(src, ax=ax, cmap=cmap, title=title)

# Plot the sentinel image 
with rasterio.open(data_file) as image_data:
    rgb = image_data.read([3, 2, 1])
    rgb_norm = normalize(rgb)
    show(rgb_norm, ax=axes[4,0], title='Image') 

# Labels 
with rasterio.open(labels_file) as labels:
    show(labels, ax=axes[4,1], cmap=cmap,  title='Labels') 

plt.tight_layout()
plt.show()